In [1]:
import numpy as np
import pandas as pd
import Soil_Carbon_Model as md
import os
import openpyxl
from pandas import *

In [2]:
dt = 1 / 24
nyears = 1000
ndays = nyears * 365
nsim = ndays * 24
nnpp = ndays * 24

#Rainfall from Marked Poisson
rain = md.poisson_process(n = int(nsim), alpha = 15.12/1000, lbd = (0.17 * dt))

tsim = np.linspace(1, nnpp, int(nnpp)) 

days = np.arange(365)
annual_avg_npp = 492.8

shape = np.sin((np.pi*days)/365)    

daily_npp = annual_avg_npp * (shape / np.sum(shape))
hourly_npp = np.repeat(daily_npp, 24)

npp = np.tile(hourly_npp,nyears)

In [3]:
print("Maximum Value is:", max(npp))
print("Average value is:", np.mean(npp))

Maximum Value is: 2.1207836723891114
Average value is: 1.3501369863013712


In [4]:
total_rainfall = rain.sum()                  # total rainfall in 'm'
print(f"Total rainfall over 450 years = {total_rainfall:.3f} meters")

RPY = (total_rainfall / nyears)*1000         # average rainfall per year in 'mm'
print(f"Average rainfall per year = {RPY:.3f} mm/year")

PET = 1260.9
AI = PET/RPY
print(AI)

Total rainfall over 450 years = 931.707 meters
Average rainfall per year = 931.707 mm/year
1.3533225213233686


In [5]:
# Compute TI_DI and TI_Texture

KA_final = 9.0132
n_final = 1.7444
R_max = 3.8                   # Max AI used for normalization

def compute_TI(AI):
    AI = np.asarray(AI, dtype=float)
    numerator   = AI**n_final
    denominator = KA_final**n_final + AI**n_final
    norm_factor = R_max**n_final / (KA_final**n_final + R_max**n_final)
    TI = (numerator / denominator) / norm_factor

    return np.minimum(TI, 1.0)

# ------------------------------------
# TI FROM SAND (SIGMOID)
# ------------------------------------
sand = 30.0   # BC sand (%)
s50 = 25.0
k   = 0.08

def TI_from_sand(sand):
    return 1.0 / (1.0 + np.exp(-k * (sand - s50)))

TI_sand = TI_from_sand(sand)
TI_AI = compute_TI(AI)
TI = (TI_sand + TI_AI)/2

print("TI_DI =", TI_AI)
print("TI_tex =", TI_sand)
print("TI Total =", TI)

TI_DI = 0.19461613179092652
TI_tex = 0.598687660112452
TI Total = 0.39665189595168926


In [6]:
#Hydraulic parameterization for dynamic porosity -------------------------------
alpha1 = 580  #hydraulic conductivity shape parameter
beta1 = 6     #hydraulic conductivity shape parameter
alpha2 = 580  #porosity shape parameter
beta2 = 6     #porosity shape parameter
kstex = 0.1   #m/d,
kstot = 1     #m/d
phitex = 0.50  
phitot = 0.60  

In [7]:
# Simulation for 2150 years at dynamic porosity
sim = md.run_julia(rain, npp, kstex, kstot, phitex, phitot, alpha1, beta1, alpha2, beta2, TI)
sim_year = sim.pivot_table(values=['s', 'phi', 'ksstr', 'P', 'DOC', 'A', 'M', 'B',
                                   'Fa', 'Fpa','Flb','Fl', 'Flm', 'Fpl', 'Fbm','Fld', 'Fal',
                                   'lk','npp_', 'L','Fi', 'Flpom', 'decomp'],
                           index='Years')

print(sim_year)

100%|█████████████████████████████████████████████████████████████████████| 8760000/8760000 [14:27<00:00, 10099.48it/s]


                 A          B         DOC        Fa       Fal       Fbm  \
Years                                                                     
1       937.990528  57.704129   61.408318  0.221221  0.005519  0.386207   
2       968.131258  11.712009  107.116014  0.224535  0.002596  0.078383   
3      1051.166530   4.598505  170.536826  0.239749  0.001408  0.030773   
4      1121.179733   4.535988  256.632692  0.247268  0.001485  0.030351   
5      1183.651114   9.320155  360.197656  0.275965  0.002910  0.062360   
...            ...        ...         ...       ...       ...       ...   
996    1753.681925  95.152887  235.529685  0.402160  0.011929  0.636714   
997    1753.508337  97.048522  222.198677  0.406199  0.012104  0.649408   
998    1753.420955  92.470756  238.820858  0.406975  0.012009  0.618769   
999    1753.219284  97.085558  233.851612  0.412393  0.012286  0.649659   
1000   1752.936532  84.898401  247.149836  0.417373  0.012093  0.568109   

             Fi        F

In [8]:
sim_year.to_csv(r"C:\Users\rishabh.singh\OneDrive - Texas A&M AgriLife\Desktop\Slope\Bhattacharya\BC.csv", index= True, header = True)
sim_year.to_csv(r"C:\Users\rishabh.singh\OneDrive - Texas A&M AgriLife\Desktop\Slope\Results\BC.csv", index= True, header = True)